# NetSentinel — Expert 6: Data Exfiltration (VAE) — High-AUC Build

**Goal**: ROC-AUC ≥ 0.85 via aggressive DNS feature engineering + ensemble scoring  
**Key insight**: DNS exfil signal lives in CHARACTER-LEVEL statistics of subdomains (entropy, n-gram patterns, vowel ratio). Native numeric columns are noise.  
**Dataset**: CIC-Bell-DNS-EXF-2021  
**Enable GPU**: Settings → Accelerator → GPU T4

In [ ]:
!pip install -q onnxruntime

In [ ]:
import os, glob, gc, json, math, time, warnings, re, string
from collections import Counter
from itertools import groupby
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix,
                             f1_score, accuracy_score)
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
print('device:', DEVICE)

EXFIL_ROOT = '/kaggle/input/cicbelldnsexf2021'
OUT_DIR = '/kaggle/working/output'
os.makedirs(OUT_DIR, exist_ok=True)

# Scan for files
all_files = sorted(glob.glob(os.path.join(EXFIL_ROOT, '**', '*.csv'), recursive=True))
if not all_files:
    # Try alternate paths
    for alt in ['/kaggle/input/datasets/humera11/cicbelldnsexf2021',
                '/kaggle/input']:
        all_files = sorted(glob.glob(os.path.join(alt, '**', '*.csv'), recursive=True))
        if all_files:
            EXFIL_ROOT = alt
            break

print(f'Found {len(all_files)} CSV files in {EXFIL_ROOT}:')
for f in all_files:
    sz = os.path.getsize(f) / (1024*1024)
    print(f'  {f}  ({sz:.1f} MB)')

## 1. Load & Label

In [ ]:
def label_from_path(path):
    """0=benign, 1=exfil. Filename first, then directory. Attack > benign priority."""
    path_lower = path.lower().replace('\\', '/')
    fn = path_lower.split('/')[-1]
    # Filename check
    if 'attack' in fn or 'exfil' in fn or 'malicious' in fn:
        return 1
    if 'benign' in fn or 'normal' in fn or 'legitimate' in fn:
        return 0
    # Directory check — attack before benign
    parent = path_lower.split('/')[-2] if '/' in path_lower else ''
    if 'attack' in parent or 'exfil' in parent:
        return 1
    if parent == 'benign' or ('benign' in parent and 'attack' not in parent):
        return 0
    # Full path
    if 'attack' in path_lower or 'exfil' in path_lower:
        return 1
    return 0

def norm_cols(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
    return df

# Load and tag
benign_frames, exfil_frames = [], []
for f in all_files:
    try:
        df = pd.read_csv(f, low_memory=False)
        df = norm_cols(df)
        label = label_from_path(f)
        tag = 'BENIGN' if label == 0 else 'EXFIL'
        print(f'  {tag:6s} | {len(df):>8,} rows | {df.shape[1]:>3} cols | {os.path.basename(f)}')
        if label == 0:
            benign_frames.append(df)
        else:
            exfil_frames.append(df)
    except Exception as e:
        print(f'  ERR: {os.path.basename(f)}: {e}')

assert benign_frames, 'No benign files found!'
assert exfil_frames, 'No exfil files found! Check label_from_path output above.'

# Find common columns
all_loaded = benign_frames + exfil_frames
common_cols = set(all_loaded[0].columns)
for df in all_loaded[1:]:
    common_cols &= set(df.columns)
common_cols = sorted(common_cols)

df_benign = pd.concat([df[common_cols] for df in benign_frames], ignore_index=True)
df_exfil = pd.concat([df[common_cols] for df in exfil_frames], ignore_index=True)

print(f'\n>>> Benign: {len(df_benign):,} rows | Exfil: {len(df_exfil):,} rows')
print(f'>>> Common columns ({len(common_cols)}): {common_cols}')

## 2. Aggressive DNS Feature Engineering
The exfil signal lives in the **character-level statistics** of subdomain strings. Base64-encoded data has very different character patterns from real domain names.

In [ ]:
VOWELS = set('aeiou')
CONSONANTS = set('bcdfghjklmnpqrstvwxyz')

def shannon_entropy(s):
    s = str(s)
    if len(s) == 0: return 0.0
    n = len(s)
    return float(-sum((c/n) * math.log2(c/n) for c in Counter(s).values()))

def bigram_entropy(s):
    s = str(s)
    if len(s) < 2: return 0.0
    bigrams = [s[i:i+2] for i in range(len(s)-1)]
    n = len(bigrams)
    return float(-sum((c/n) * math.log2(c/n) for c in Counter(bigrams).values()))

def trigram_entropy(s):
    s = str(s)
    if len(s) < 3: return 0.0
    trigrams = [s[i:i+3] for i in range(len(s)-2)]
    n = len(trigrams)
    return float(-sum((c/n) * math.log2(c/n) for c in Counter(trigrams).values()))

def max_consonant_run(s):
    s = str(s).lower()
    max_run = 0
    current = 0
    for c in s:
        if c in CONSONANTS:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0
    return max_run

def find_dns_col(df):
    for key in ['subdomain', 'domain', 'query', 'fqdn', 'hostname', 'host',
                'longest_word', 'sld', 'url', 'name']:
        for c in df.columns:
            if key in c and df[c].dtype == object:
                return c
    # fallback: longest average string column
    str_cols = [c for c in df.columns if df[c].dtype == object]
    if str_cols:
        avg_lens = {c: df[c].astype(str).str.len().mean() for c in str_cols}
        return max(avg_lens, key=avg_lens.get)
    return None

def engineer_dns_features(df):
    """Extract 20+ character-level features from the DNS string column."""
    dns_col = find_dns_col(df)
    if dns_col is None:
        print('WARNING: No DNS string column found!')
        return pd.DataFrame()
    
    print(f'  Engineering features from: {dns_col!r}')
    s = df[dns_col].astype(str).fillna('')
    L = s.str.len().astype('float32')
    Lp = L + 1.0  # prevent div by zero
    
    feats = pd.DataFrame()
    
    # === Length features ===
    feats['dns_len'] = L
    feats['dns_log_len'] = np.log1p(L)
    feats['dns_label_count'] = (s.str.count(r'\.') + 1).astype('float32')
    feats['dns_longest_token'] = s.str.split(r'[.\-_]').map(
        lambda parts: max((len(p) for p in parts), default=0)).astype('float32')
    feats['dns_avg_token_len'] = s.str.split(r'[.\-_]').map(
        lambda parts: np.mean([len(p) for p in parts]) if parts else 0).astype('float32')
    
    # === Entropy features (THE key exfil signal) ===
    feats['dns_entropy'] = s.map(shannon_entropy).astype('float32')
    feats['dns_bigram_entropy'] = s.map(bigram_entropy).astype('float32')
    feats['dns_trigram_entropy'] = s.map(trigram_entropy).astype('float32')
    # Normalized entropy (entropy / log2(len)) — pure randomness measure
    feats['dns_norm_entropy'] = (feats['dns_entropy'] / np.log2(Lp + 1)).astype('float32')
    
    # === Character class ratios ===
    feats['dns_digit_ratio'] = (s.str.count(r'[0-9]') / Lp).astype('float32')
    feats['dns_upper_ratio'] = (s.str.count(r'[A-Z]') / Lp).astype('float32')
    feats['dns_lower_ratio'] = (s.str.count(r'[a-z]') / Lp).astype('float32')
    feats['dns_alpha_ratio'] = (s.str.count(r'[A-Za-z]') / Lp).astype('float32')
    feats['dns_special_ratio'] = (s.str.count(r'[^A-Za-z0-9.]') / Lp).astype('float32')
    feats['dns_hex_ratio'] = (s.str.count(r'[0-9a-fA-F]') / Lp).astype('float32')
    
    # === Linguistic features (real domains have vowels, base64 doesn't) ===
    feats['dns_vowel_ratio'] = (s.str.lower().map(
        lambda x: sum(1 for c in x if c in VOWELS)) / Lp).astype('float32')
    feats['dns_consonant_ratio'] = (s.str.lower().map(
        lambda x: sum(1 for c in x if c in CONSONANTS)) / Lp).astype('float32')
    feats['dns_vowel_consonant_ratio'] = (
        feats['dns_vowel_ratio'] / (feats['dns_consonant_ratio'] + 0.01)).astype('float32')
    feats['dns_max_consonant_run'] = s.map(max_consonant_run).astype('float32')
    
    # === Repetition / uniqueness ===
    feats['dns_unique_chars'] = s.map(lambda x: len(set(x))).astype('float32')
    feats['dns_unique_ratio'] = (feats['dns_unique_chars'] / Lp).astype('float32')
    feats['dns_max_repeat'] = s.map(
        lambda x: max((sum(1 for _ in g) for _, g in groupby(x)), default=0)).astype('float32')
    
    # === Also keep native numeric features that might help ===
    for c in df.columns:
        if c == dns_col or c in feats.columns:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            feats[f'native_{c}'] = pd.to_numeric(df[c], errors='coerce').astype('float32')
        elif df[c].dtype == object:
            conv = pd.to_numeric(df[c], errors='coerce')
            if conv.notna().mean() > 0.9:
                feats[f'native_{c}'] = conv.astype('float32')
    
    return feats

print('=== Benign ===')
F_benign = engineer_dns_features(df_benign)
print(f'  → {F_benign.shape[1]} features, {len(F_benign):,} rows')

print('\n=== Exfil ===')
F_exfil = engineer_dns_features(df_exfil)
print(f'  → {F_exfil.shape[1]} features, {len(F_exfil):,} rows')

# Keep only common features
common_feats = sorted(set(F_benign.columns) & set(F_exfil.columns))
F_benign = F_benign[common_feats]
F_exfil = F_exfil[common_feats]
print(f'\n>>> {len(common_feats)} common features: {common_feats}')

## 3. Feature Discrimination Check
Verify signal exists BEFORE training.

In [ ]:
# Sample for speed
n_check = min(10000, len(F_benign), len(F_exfil))
X_chk = np.vstack([F_benign.sample(n_check, random_state=SEED).values,
                    F_exfil.sample(n_check, random_state=SEED).values])
y_chk = np.array([0]*n_check + [1]*n_check)

print('Feature discrimination (univariate AUC):')
print(f'{"Feature":40s} {"AUC":>8s} {"Signal":>8s}')
print('-' * 60)
feat_scores = []
for i, c in enumerate(common_feats):
    col = X_chk[:, i]
    valid = np.isfinite(col)
    if valid.sum() < 100:
        continue
    auc = roc_auc_score(y_chk[valid], col[valid])
    sep = abs(auc - 0.5)
    feat_scores.append((c, auc, sep))

feat_scores.sort(key=lambda t: t[2], reverse=True)
for c, auc, sep in feat_scores:
    marker = ' ★★★' if sep > 0.2 else ' ★★' if sep > 0.1 else ' ★' if sep > 0.05 else ''
    print(f'  {c:38s} {auc:8.4f} {sep:8.4f}{marker}')

# Keep features with |AUC-0.5| > 0.02
strong_feats = [c for c, auc, sep in feat_scores if sep > 0.02]
print(f'\n>>> Keeping {len(strong_feats)} features with signal > 0.02')

# Always keep dns_ features
for c in common_feats:
    if c.startswith('dns_') and c not in strong_feats:
        strong_feats.append(c)
strong_feats = sorted(set(strong_feats))
print(f'>>> Final feature count: {len(strong_feats)}')

## 4. Build Matrices + Split

In [ ]:
X_b = F_benign[strong_feats].values.astype('float32')
X_e = F_exfil[strong_feats].values.astype('float32')

# Clean inf/nan
X_b = np.nan_to_num(X_b, nan=0.0, posinf=0.0, neginf=0.0)
X_e = np.nan_to_num(X_e, nan=0.0, posinf=0.0, neginf=0.0)

print(f'Benign: {X_b.shape}  |  Exfil: {X_e.shape}')

# Split benign: 70% train, 15% val, 15% test
rng = np.random.default_rng(SEED)
idx_b = rng.permutation(len(X_b))
n1 = int(0.70 * len(X_b))
n2 = int(0.85 * len(X_b))
Xtr = X_b[idx_b[:n1]]
Xvl = X_b[idx_b[n1:n2]]
Xte_b = X_b[idx_b[n2:]]

# Split exfil: 50% selection, 50% test
idx_e = rng.permutation(len(X_e))
half = len(X_e) // 2
Xsel_e = X_e[idx_e[:half]]
Xte_e = X_e[idx_e[half:]]

# Scale (fit on train benign ONLY)
scaler = RobustScaler().fit(Xtr)
def sc(X): return np.clip(scaler.transform(X), -10, 10).astype('float32')

Xtr_s = sc(Xtr)
Xvl_s = sc(Xvl)

# Selection set = val benign + selection exfil
Xsel_s = np.vstack([sc(Xvl), sc(Xsel_e)])
ysel = np.array([0]*len(Xvl) + [1]*len(Xsel_e))

# Test set = test benign + test exfil
Xte_s = np.vstack([sc(Xte_b), sc(Xte_e)])
yte = np.array([0]*len(Xte_b) + [1]*len(Xte_e))

N_FEAT = len(strong_feats)
print(f'Train: {Xtr_s.shape} (benign only)')
print(f'Selection: {Xsel_s.shape} (B:{len(Xvl)}, E:{len(Xsel_e)})')
print(f'Test: {Xte_s.shape} (B:{len(Xte_b)}, E:{len(Xte_e)})')
print(f'Features: {N_FEAT}')

## 5. VAE + Isolation Forest + Mahalanobis Ensemble

In [ ]:
# =================== VAE ===================
class VAE(nn.Module):
    def __init__(self, d_in, d_hid=256, d_lat=16):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.BatchNorm1d(d_hid), nn.LeakyReLU(0.2), nn.Dropout(0.1),
            nn.Linear(d_hid, d_hid//2), nn.BatchNorm1d(d_hid//2), nn.LeakyReLU(0.2),
        )
        self.mu = nn.Linear(d_hid//2, d_lat)
        self.lv = nn.Linear(d_hid//2, d_lat)
        self.dec = nn.Sequential(
            nn.Linear(d_lat, d_hid//2), nn.BatchNorm1d(d_hid//2), nn.LeakyReLU(0.2),
            nn.Linear(d_hid//2, d_hid), nn.BatchNorm1d(d_hid), nn.LeakyReLU(0.2),
            nn.Linear(d_hid, d_in),
        )
    def forward(self, x):
        h = self.enc(x)
        mu, lv = self.mu(h), self.lv(h)
        z = mu + torch.randn_like(mu) * torch.exp(0.5*lv)
        return self.dec(z), mu, lv
    @torch.no_grad()
    def encode_mu(self, x):
        self.eval()
        return self.mu(self.enc(x))
    @torch.no_grad()
    def reconstruct(self, x):
        self.eval()
        return self.dec(self.mu(self.enc(x)))

def vae_loss(xr, x, mu, lv, beta=1.0):
    rec = nn.functional.mse_loss(xr, x, reduction='none').sum(1)
    kld = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp(), dim=1)
    return (rec + beta*kld).mean()

model = VAE(N_FEAT, d_hid=256, d_lat=16).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
loader = DataLoader(TensorDataset(torch.tensor(Xtr_s)), batch_size=512, shuffle=True)

print(model)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# =================== Train VAE ===================
EPOCHS, PAT, WARMUP = 80, 12, 10
hist = {'tl': [], 'vl': [], 'auc': []}
best_vl, best_st, wait = 1e9, None, 0
t0 = time.time()

@torch.no_grad()
def recon_err(Xs):
    model.eval(); out = []
    for i in range(0, len(Xs), 4096):
        xb = torch.tensor(Xs[i:i+4096], dtype=torch.float32, device=DEVICE)
        out.append(((model.reconstruct(xb) - xb)**2).sum(1).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def get_latent(Xs):
    model.eval(); out = []
    for i in range(0, len(Xs), 4096):
        xb = torch.tensor(Xs[i:i+4096], dtype=torch.float32, device=DEVICE)
        out.append(model.encode_mu(xb).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def val_loss_fn(Xs):
    model.eval(); tot = 0
    for i in range(0, len(Xs), 4096):
        xb = torch.tensor(Xs[i:i+4096], dtype=torch.float32, device=DEVICE)
        xr,mu,lv = model(xb); tot += vae_loss(xr,xb,mu,lv).item()*len(xb)
    return tot / len(Xs)

for ep in range(EPOCHS):
    beta = min(1.0, (ep+1)/WARMUP)
    model.train(); tot = 0
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        xr,mu,lv = model(xb); loss = vae_loss(xr,xb,mu,lv,beta=beta)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()*len(xb)
    tl = tot/len(loader.dataset)
    vl = val_loss_fn(Xvl_s)
    sched.step(vl)
    
    se = recon_err(Xsel_s)
    auc = roc_auc_score(ysel, se)
    if auc < 0.5: auc = 1 - auc  # auto-flip
    
    hist['tl'].append(tl); hist['vl'].append(vl); hist['auc'].append(auc)
    if (ep+1) % 5 == 0 or ep == 0:
        print(f'E{ep+1:02d} TL:{tl:.4f} VL:{vl:.4f} β:{beta:.2f} AUC:{auc:.4f}')
    if vl < best_vl - 1e-4:
        best_vl = vl; best_st = {k:v.cpu().clone() for k,v in model.state_dict().items()}; wait = 0
    else:
        wait += 1
        if wait >= PAT: print(f'Early stop E{ep+1}'); break

if best_st: model.load_state_dict(best_st)
train_time = time.time() - t0
print(f'Done {train_time/60:.1f}min, best VL: {best_vl:.4f}')

In [ ]:
# =================== Build ensemble scorers ===================

# 1) VAE reconstruction error
def score_vae_recon(Xs): return recon_err(Xs)

# 2) VAE latent-space Mahalanobis (anomaly = far from benign cluster in latent space)
Z_benign = get_latent(Xtr_s)
try:
    lat_cov = EmpiricalCovariance().fit(Z_benign)
    def score_vae_latent(Xs): return lat_cov.mahalanobis(get_latent(Xs))
    print('Latent Mahalanobis: OK')
except:
    def score_vae_latent(Xs): return recon_err(Xs)
    print('Latent Mahalanobis: fallback to recon')

# 3) Isolation Forest on input features
n_if = min(50000, len(Xtr_s))
iso = IsolationForest(n_estimators=300, contamination='auto', random_state=SEED, n_jobs=-1)
iso.fit(Xtr_s[rng.choice(len(Xtr_s), n_if, replace=False)])
def score_iso(Xs): return -iso.score_samples(Xs)
print('Isolation Forest: OK')

# 4) Input-space Mahalanobis on DNS features only
dns_idx = [i for i, c in enumerate(strong_feats) if c.startswith('dns_')]
if dns_idx:
    try:
        dns_cov = EmpiricalCovariance().fit(Xtr_s[:, dns_idx])
        def score_dns_maha(Xs): return dns_cov.mahalanobis(Xs[:, dns_idx])
        print(f'DNS Mahalanobis: OK ({len(dns_idx)} features)')
    except:
        score_dns_maha = None
        print('DNS Mahalanobis: failed')
else:
    score_dns_maha = None

# =================== Score each on selection set ===================
SCORERS = {
    'vae_recon': score_vae_recon,
    'vae_latent': score_vae_latent,
    'iso': score_iso,
}
if score_dns_maha is not None:
    SCORERS['dns_maha'] = score_dns_maha

sel_aucs = {}
for name, fn in SCORERS.items():
    raw = fn(Xsel_s)
    auc = roc_auc_score(ysel, raw)
    if auc < 0.5:
        auc = 1 - auc  # flip
    sel_aucs[name] = auc

# Fusion: z-normalize + average
val_stats = {}
for name, fn in SCORERS.items():
    v = fn(Xvl_s)
    val_stats[name] = (v.mean(), v.std() + 1e-9)

def score_fusion(Xs):
    scores = []
    for name, fn in SCORERS.items():
        raw = fn(Xs)
        m, s = val_stats[name]
        z = (raw - m) / s
        # Check if this scorer needs flipping
        if sel_aucs[name] < 0.5:  # was flipped
            z = -z
        scores.append(z)
    return np.mean(scores, axis=0)

fus_auc = roc_auc_score(ysel, score_fusion(Xsel_s))
if fus_auc < 0.5: fus_auc = 1 - fus_auc
sel_aucs['fusion'] = fus_auc

print('\n=== Selection-Set ROC-AUC per Scorer ===')
for name, auc in sorted(sel_aucs.items(), key=lambda t: -t[1]):
    print(f'  {name:15s} {auc:.4f}{" ← BEST" if auc == max(sel_aucs.values()) else ""}')

# Auto-select best
BEST = max(sel_aucs, key=sel_aucs.get)
all_scorers = dict(SCORERS)
all_scorers['fusion'] = score_fusion
anomaly_score = all_scorers[BEST]
print(f'\n>>> Selected: {BEST} (AUC={sel_aucs[BEST]:.4f})')

## 6. Evaluate on Test Set

In [ ]:
# Score test set
test_scores = anomaly_score(Xte_s)
val_scores = anomaly_score(Xvl_s)  # benign only for threshold

# Check if flip needed
raw_test_auc = roc_auc_score(yte, test_scores)
FLIP = raw_test_auc < 0.5
if FLIP:
    test_scores = -test_scores
    val_scores = -val_scores
    print(f'Scores flipped (raw AUC was {raw_test_auc:.4f})')

# Threshold at FPR budget
FPR_BUDGET = 0.01
thr = float(np.percentile(val_scores, 100 * (1 - FPR_BUDGET)))
pred = (test_scores > thr).astype(int)

# Also try optimal threshold (max F1)
from sklearn.metrics import precision_recall_curve
prec_arr, rec_arr, thr_arr = precision_recall_curve(yte, test_scores)
f1_arr = 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-9)
best_f1_idx = np.argmax(f1_arr)
best_thr = thr_arr[min(best_f1_idx, len(thr_arr)-1)]
pred_best = (test_scores > best_thr).astype(int)

roc = roc_auc_score(yte, test_scores)
prc = average_precision_score(yte, test_scores)

print(f'\n{"="*60}')
print(f'  EXPERT 6: DATA EXFILTRATION — RESULTS (scorer: {BEST})')
print(f'{"="*60}')
print(f'  ROC-AUC:            {roc:.4f}')
print(f'  PR-AUC:             {prc:.4f}')
print(f'  Score flipped:      {FLIP}')
print(f'')
print(f'  --- At FPR Budget ({FPR_BUDGET:.0%}) ---')
print(f'  F1:                 {f1_score(yte, pred):.4f}')
print(f'  Accuracy:           {accuracy_score(yte, pred):.4f}')
print(f'  Recall @ {FPR_BUDGET:.0%} FPR:  {float((test_scores[yte==1] > thr).mean()):.4f}')
print(f'')
print(f'  --- At Optimal Threshold (max F1) ---')
print(f'  Best F1:            {f1_arr[best_f1_idx]:.4f}')
print(f'  Accuracy:           {accuracy_score(yte, pred_best):.4f}')
print(f'{"="*60}')
print()
print('Classification Report (optimal threshold):')
print(classification_report(yte, pred_best, target_names=['Benign', 'Exfil']))

## 7. Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(hist['tl'], label='Train', lw=2)
axes[0,0].plot(hist['vl'], label='Val', lw=2)
axes[0,0].set_title('VAE Loss', fontweight='bold'); axes[0,0].legend(); axes[0,0].grid(alpha=.3)

axes[0,1].plot(hist['auc'], color='green', lw=2)
axes[0,1].axhline(0.5, ls='--', color='gray', alpha=.5)
axes[0,1].set_title('Selection ROC-AUC (VAE)', fontweight='bold')
axes[0,1].set_ylim(0, 1.02); axes[0,1].grid(alpha=.3)

cm = confusion_matrix(yte, pred_best)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Reds',
            xticklabels=['Benign','Exfil'], yticklabels=['Benign','Exfil'], ax=axes[1,0])
axes[1,0].set_title(f'Confusion (F1={f1_arr[best_f1_idx]:.4f})', fontweight='bold')
axes[1,0].set_ylabel('Actual'); axes[1,0].set_xlabel('Predicted')

lo, hi = np.percentile(test_scores, [1, 99])
bins = np.linspace(lo, hi, 80)
axes[1,1].hist(test_scores[yte==0], bins=bins, alpha=.6, label='Benign', density=True, color='steelblue')
axes[1,1].hist(test_scores[yte==1], bins=bins, alpha=.6, label='Exfil', density=True, color='red')
axes[1,1].axvline(best_thr, ls='--', color='k', label='Optimal threshold')
axes[1,1].set_title('Score Distribution', fontweight='bold'); axes[1,1].legend()

plt.suptitle(f'Expert 6: Data Exfiltration ({BEST})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'expert6_graphs.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Export

In [ ]:
import joblib

# Save VAE
torch.save(model.cpu().state_dict(), os.path.join(OUT_DIR, 'expert6_vae.pt'))

# ONNX export (deterministic path)
class DetRecon(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m.reconstruct(x)

det = DetRecon(model).eval()
dummy = torch.randn(1, N_FEAT)
onnx_path = os.path.join(OUT_DIR, 'expert6_vae.onnx')
torch.onnx.export(det, dummy, onnx_path,
    input_names=['dns_features'], output_names=['reconstruction'],
    dynamic_axes={'dns_features':{0:'batch'}, 'reconstruction':{0:'batch'}},
    opset_version=17)
print(f'ONNX: {os.path.getsize(onnx_path)/1024:.0f} KB')

# Save ensemble artifacts
joblib.dump(scaler, os.path.join(OUT_DIR, 'expert6_scaler.joblib'))
joblib.dump(iso, os.path.join(OUT_DIR, 'expert6_iso.joblib'))
joblib.dump({'selected': BEST, 'sel_aucs': sel_aucs, 'flip': FLIP,
             'val_stats': val_stats, 'dns_idx': dns_idx,
             'feature_names': strong_feats},
            os.path.join(OUT_DIR, 'expert6_ensemble.joblib'))

# Metrics JSON
metrics = {
    'model_name': 'NetSentinel Expert 6: Data Exfiltration',
    'model_type': f'Unsupervised Ensemble (selected: {BEST})',
    'version': '2.0.0',
    'roc_auc': float(roc), 'pr_auc': float(prc),
    'best_f1': float(f1_arr[best_f1_idx]),
    'accuracy_at_best_f1': float(accuracy_score(yte, pred_best)),
    'fpr_budget': FPR_BUDGET, 'score_flipped': FLIP,
    'n_features': N_FEAT, 'feature_names': strong_feats,
    'scorer_aucs': {k: float(v) for k, v in sel_aucs.items()},
    'mitre': {'T1041': 'Exfil Over C2', 'T1048': 'Exfil Alt Protocol', 'T1071.004': 'DNS'},
    'dataset': 'CIC-Bell-DNS-EXF-2021'
}
json.dump(metrics, open(os.path.join(OUT_DIR, 'expert6_meta.json'), 'w'), indent=2)

print(f'\nModel Card:')
print(f'  ROC-AUC:   {roc:.4f}')
print(f'  PR-AUC:    {prc:.4f}')
print(f'  Best F1:   {f1_arr[best_f1_idx]:.4f}')
print(f'  Scorer:    {BEST}')
print(f'  Features:  {N_FEAT}')

In [ ]:
import zipfile
from IPython.display import FileLink

zip_path = '/kaggle/working/netsentinel_expert6.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT_DIR):
        fp = os.path.join(OUT_DIR, f)
        if os.path.isfile(fp):
            zf.write(fp, f)
            print(f'  {f:40s} ({os.path.getsize(fp)/1024:.1f} KB)')
print(f'\nZip: {os.path.getsize(zip_path)/1024/1024:.1f} MB')
FileLink(zip_path)